# 🏀 Defense Decoded: Quantifying Basketball's Dark Matter

> **Defense has always been basketball's dark matter.** Using tracking, hustle, and synergy data unavailable in any other public dataset, we build the most complete defensive player profiles ever assembled.

**Part 5 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

## What makes this analysis unique?

Most NBA datasets give you box-score stats: steals, blocks, defensive rebounds. But real defense is invisible in those numbers. This dataset uniquely combines:

1. **Tracking defense** — opponent FG% impact by defender (how much worse do shooters perform when guarded by this player?)
2. **Hustle stats** — contested shots, deflections, charges drawn, loose balls recovered (effort plays the box score misses)
3. **Synergy defensive play types** — PPP allowed by play type (isolation, pick-and-roll, post-up, etc.)

We fuse all three into **Defensive DNA profiles** using PCA and build a composite **Defensive Impact Score** validated against on/off net rating.

---

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Tracking Defense: Opponent FG% Impact](#2-tracking-defense-opponent-fg-impact)
3. [Hustle Stats: The Effort Dimension](#3-hustle-stats-the-effort-dimension)
4. [Synergy Defense: Play Type Efficiency](#4-synergy-defense-play-type-efficiency)
5. [Defensive DNA Profiles (PCA)](#5-defensive-dna-profiles-pca)
6. [Defensive Impact Score](#6-defensive-impact-score)
7. [Two-Way Elite Scatter](#7-two-way-elite-scatter)
8. [Era Trends: Defensive Evolution](#8-era-trends-defensive-evolution)
9. [Conclusion & Cross-Links](#9-conclusion--cross-links)

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2"

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import duckdb
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from pathlib import Path

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, TEMPLATE, takeaway, get_connection

conn = get_connection()
print("Setup complete.")

---
## 2. Tracking Defense: Opponent FG% Impact

The `fact_tracking_defense` table captures how opponents shoot when guarded by each
defender, broken down by `defense_category` (Overall, 3 Pointers, 2 Pointers, ≤ 6ft,
≥ 15ft, etc.). The key metric is **`pct_plusminus`** — the difference between
the opponent's FG% when guarded by this player vs their normal FG%. A negative value
means the defender is making opponents shoot *worse* than their average.

In [ ]:
# Explore tracking defense schema and categories
td_schema = conn.sql("SELECT * FROM fact_tracking_defense LIMIT 5").pl()
print("Columns:", td_schema.columns)
print()

categories = conn.sql("""
    SELECT DISTINCT defense_category, COUNT(*) AS n
    FROM fact_tracking_defense
    GROUP BY defense_category
    ORDER BY n DESC
""").pl()
print("Defense categories:")
print(categories)

In [ ]:
# Aggregate tracking defense to player-season level (Overall category)
tracking_def = conn.sql("""
    SELECT
        td.player_id,
        p.full_name AS player_name,
        p.position,
        td.season_year,
        AVG(td.d_fga) AS avg_d_fga,
        AVG(td.d_fg_pct) AS avg_d_fg_pct,
        AVG(td.normal_fg_pct) AS avg_normal_fg_pct,
        AVG(td.pct_plusminus) AS avg_pct_plusminus,
        SUM(td.d_fga) AS total_d_fga
    FROM fact_tracking_defense td
    JOIN dim_player p ON td.player_id = p.player_id
    WHERE td.defense_category = 'Overall'
    GROUP BY td.player_id, p.full_name, p.position, td.season_year
    HAVING SUM(td.d_fga) >= 200
    ORDER BY avg_pct_plusminus ASC
""").pl()

print(f"Player-seasons with tracking defense (Overall, 200+ DFGA): {len(tracking_def):,}")
print(f"Seasons: {tracking_def['season_year'].min()} - {tracking_def['season_year'].max()}")

In [ ]:
# Distribution of pct_plusminus
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=tracking_def["avg_pct_plusminus"].to_list(),
    nbinsx=60,
    marker_color=COLORS["primary"],
    opacity=0.8,
    hovertemplate="FG% +/-: %{x:.1f}%<br>Count: %{y}<extra></extra>",
))
fig.add_vline(x=0, line_dash="dash", line_color=COLORS["muted"],
              annotation_text="League avg")
fig.update_layout(
    title="Distribution of Opponent FG% Impact (pct_plusminus)",
    xaxis_title="Opponent FG% +/- (negative = better defender)",
    yaxis_title="Player-Seasons",
    height=450,
)
fig.show()

In [ ]:
# Top-20 defenders by pct_plusminus (most negative = best)
top20_tracking = tracking_def.head(20)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Rank", "Player", "Position", "Season", "DFGA",
                "Opp FG%", "Normal FG%", "FG% +/-"],
        fill_color=COLORS["primary"],
        font=dict(color="white", size=12),
        align="center",
    ),
    cells=dict(
        values=[
            list(range(1, 21)),
            top20_tracking["player_name"].to_list(),
            top20_tracking["position"].to_list(),
            top20_tracking["season_year"].to_list(),
            [f"{v:.0f}" for v in top20_tracking["total_d_fga"].to_list()],
            [f"{v:.1f}%" for v in top20_tracking["avg_d_fg_pct"].to_list()],
            [f"{v:.1f}%" for v in top20_tracking["avg_normal_fg_pct"].to_list()],
            [f"{v:+.1f}%" for v in top20_tracking["avg_pct_plusminus"].to_list()],
        ],
        fill_color="white",
        align="center",
        font=dict(size=11),
    ),
)])
fig.update_layout(
    title="Top 20 Defenders by Opponent FG% Impact (Lower = Better)",
    height=550, width=850,
)
fig.show()

takeaway(
    "The best defenders in tracking data reduce opponent FG% by 4-8 percentage points "
    "below their normal shooting. This is the most direct measurement of individual "
    "defensive impact available and captures what steals and blocks cannot: the shots "
    "opponents simply miss because of the defender's presence."
)

---
## 3. Hustle Stats: The Effort Dimension

Hustle stats measure the *effort* plays that never show up in the box score:
contested shots, deflections, charges drawn, loose balls recovered, screen assists,
and box outs. These capture the invisible work that separates good defenders from
great ones.

In [ ]:
# Aggregate hustle stats to player-season per-game averages
hustle_season = conn.sql("""
    SELECT
        h.player_id,
        p.full_name AS player_name,
        p.position,
        g.season_year,
        COUNT(*) AS games_played,
        AVG(h.contested_shots) AS contested_shots_pg,
        AVG(h.contested_shots_2pt) AS contested_shots_2pt_pg,
        AVG(h.contested_shots_3pt) AS contested_shots_3pt_pg,
        AVG(h.deflections) AS deflections_pg,
        AVG(h.charges_drawn) AS charges_drawn_pg,
        AVG(h.screen_assists) AS screen_assists_pg,
        AVG(h.screen_ast_pts) AS screen_ast_pts_pg,
        AVG(h.loose_balls_recovered) AS loose_balls_recovered_pg,
        AVG(h.box_outs) AS box_outs_pg
    FROM fact_player_game_hustle h
    JOIN dim_game g ON h.game_id = g.game_id
    JOIN dim_player p ON h.player_id = p.player_id
    WHERE g.season_type = 'Regular Season'
    GROUP BY h.player_id, p.full_name, p.position, g.season_year
    HAVING COUNT(*) >= 40
    ORDER BY contested_shots_pg DESC
""").pl()

print(f"Player-seasons with hustle data (40+ GP): {len(hustle_season):,}")
print(f"Seasons: {hustle_season['season_year'].min()} - {hustle_season['season_year'].max()}")

In [ ]:
# Top-15 leaders in each hustle category
hustle_cats = [
    ("contested_shots_pg", "Contested Shots / Game"),
    ("contested_shots_2pt_pg", "Contested 2PT Shots / Game"),
    ("contested_shots_3pt_pg", "Contested 3PT Shots / Game"),
    ("deflections_pg", "Deflections / Game"),
    ("charges_drawn_pg", "Charges Drawn / Game"),
    ("loose_balls_recovered_pg", "Loose Balls Recovered / Game"),
]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[title for _, title in hustle_cats],
    vertical_spacing=0.10,
    horizontal_spacing=0.12,
)

bar_colors = [COLORS["primary"], COLORS["secondary"], COLORS["accent"],
              COLORS["success"], COLORS["teal"], COLORS["purple"]]

for idx, (col, title) in enumerate(hustle_cats):
    row = idx // 2 + 1
    col_idx = idx % 2 + 1
    top15 = hustle_season.sort(col, descending=True).head(15)
    labels = [
        f"{n} ({s})"
        for n, s in zip(top15["player_name"].to_list(), top15["season_year"].to_list())
    ]
    fig.add_trace(
        go.Bar(
            x=top15[col].to_list(),
            y=labels,
            orientation="h",
            marker_color=bar_colors[idx],
            showlegend=False,
            hovertemplate="%{y}<br>%{x:.2f}<extra></extra>",
        ),
        row=row, col=col_idx,
    )
    fig.update_yaxes(autorange="reversed", row=row, col=col_idx)

fig.update_layout(
    height=1200, width=1000,
    title_text="Hustle Stat Leaders by Category (Per Game, 40+ GP Seasons)",
)
fig.show()

takeaway(
    "Hustle stats reveal a completely different defensive hierarchy than blocks and steals. "
    "The league leaders in contested shots are often bigs who rarely lead in steals, while "
    "deflection leaders tend to be long-armed wings and guards. Charges drawn are dominated "
    "by a small set of high-IQ defenders who consistently put their bodies on the line "
    "-- a skill that does not correlate strongly with any physical attribute."
)

---
## 4. Synergy Defense: Play Type Efficiency

Synergy tracks every possession by play type. The `fact_synergy` table includes
defensive data with `type_grouping` indicating defensive play types. The key metric
is **PPP (points per possession)** allowed — lower is better for a defender.

In [ ]:
# Explore synergy defensive data
synergy_types = conn.sql("""
    SELECT DISTINCT type_grouping, entity_type, play_type, COUNT(*) AS n
    FROM fact_synergy
    WHERE type_grouping ILIKE '%defensive%'
       OR type_grouping ILIKE '%defense%'
    GROUP BY type_grouping, entity_type, play_type
    ORDER BY type_grouping, n DESC
""").pl()
print("Synergy defensive play types:")
print(synergy_types)

In [ ]:
# Defensive PPP by play type across the league
synergy_def = conn.sql("""
    SELECT
        play_type,
        COUNT(DISTINCT player_id) AS n_players,
        AVG(ppp) AS avg_ppp,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY ppp) AS p25_ppp,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY ppp) AS median_ppp,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY ppp) AS p75_ppp,
        AVG(poss) AS avg_poss
    FROM fact_synergy
    WHERE (type_grouping ILIKE '%defensive%' OR type_grouping ILIKE '%defense%')
      AND poss >= 20
    GROUP BY play_type
    HAVING COUNT(DISTINCT player_id) >= 50
    ORDER BY avg_ppp ASC
""").pl()

print(f"Defensive play types with sufficient data: {len(synergy_def)}")
print(synergy_def)

In [ ]:
# League-wide defensive PPP distribution by play type
fig = go.Figure()

play_types = synergy_def["play_type"].to_list()
fig.add_trace(go.Bar(
    x=play_types,
    y=synergy_def["avg_ppp"].to_list(),
    name="Avg PPP Allowed",
    marker_color=COLORS["secondary"],
    error_y=dict(
        type="data",
        symmetric=False,
        array=(synergy_def["p75_ppp"] - synergy_def["avg_ppp"]).to_list(),
        arrayminus=(synergy_def["avg_ppp"] - synergy_def["p25_ppp"]).to_list(),
        color=COLORS["muted"],
    ),
    hovertemplate="%{x}<br>Avg PPP: %{y:.3f}<extra></extra>",
))

fig.update_layout(
    title="Defensive PPP Allowed by Play Type (League Average)",
    xaxis_title="Defensive Play Type",
    yaxis_title="Points Per Possession Allowed",
    height=500,
    xaxis_tickangle=-30,
)
fig.show()

In [ ]:
# Top defenders by play type: show best PPP allowed for major types
major_types = conn.sql("""
    SELECT play_type
    FROM fact_synergy
    WHERE (type_grouping ILIKE '%defensive%' OR type_grouping ILIKE '%defense%')
      AND poss >= 50
    GROUP BY play_type
    HAVING COUNT(DISTINCT player_id) >= 100
    ORDER BY COUNT(*) DESC
    LIMIT 5
""").pl()["play_type"].to_list()

top_def_by_type = conn.sql(f"""
    SELECT
        s.player_id,
        p.full_name AS player_name,
        s.play_type,
        s.season_year,
        s.ppp,
        s.poss,
        s.percentile
    FROM fact_synergy s
    JOIN dim_player p ON s.player_id = p.player_id
    WHERE (s.type_grouping ILIKE '%defensive%' OR s.type_grouping ILIKE '%defense%')
      AND s.poss >= 50
      AND s.play_type IN ({','.join(f"'{t}'" for t in major_types)})
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY s.play_type ORDER BY s.ppp ASC
    ) <= 10
    ORDER BY s.play_type, s.ppp ASC
""").pl()

# Grouped bar: top 10 defenders per major play type
fig = make_subplots(
    rows=len(major_types), cols=1,
    subplot_titles=[f"Best Defenders: {pt}" for pt in major_types],
    vertical_spacing=0.06,
)

for idx, pt in enumerate(major_types):
    subset = top_def_by_type.filter(pl.col("play_type") == pt)
    labels = [
        f"{n} ({s})"
        for n, s in zip(subset["player_name"].to_list(), subset["season_year"].to_list())
    ]
    fig.add_trace(
        go.Bar(
            x=subset["ppp"].to_list(),
            y=labels,
            orientation="h",
            marker_color=bar_colors[idx % len(bar_colors)],
            showlegend=False,
            hovertemplate="%{y}<br>PPP: %{x:.3f}<extra></extra>",
        ),
        row=idx + 1, col=1,
    )
    fig.update_yaxes(autorange="reversed", row=idx + 1, col=1)

fig.update_layout(
    height=350 * len(major_types),
    width=900,
    title_text="Top 10 Defenders by PPP Allowed (Major Play Types, 50+ Poss)",
)
fig.show()

takeaway(
    "Synergy play-type data reveals defender specialization invisible in aggregate stats. "
    "Some players are elite isolation defenders but average in pick-and-roll coverage, "
    "while others specialize in post defense. The best overall defenders tend to be "
    "versatile across multiple play types, which is what makes them so valuable."
)

---
## 5. Defensive DNA Profiles (PCA)

Now we fuse all three data sources — tracking defense, hustle stats, and synergy —
into a single defensive feature matrix per player-season. We use PCA to discover
the latent dimensions of defense and interpret them as archetypes:

- **Component 1 — Rim Protection**: contested 2PT shots, box outs, blocks
- **Component 2 — Perimeter Lock**: contested 3PT shots, deflections, FG% impact
- **Component 3 — Effort/IQ**: loose balls recovered, charges drawn, screen assists

In [ ]:
# Join tracking defense + hustle + traditional stats for the feature matrix
# We also grab blocks and steals from agg_player_season
defense_matrix = conn.sql("""
    WITH tracking AS (
        SELECT
            td.player_id,
            td.season_year,
            AVG(td.pct_plusminus) AS pct_plusminus
        FROM fact_tracking_defense td
        WHERE td.defense_category = 'Overall'
        GROUP BY td.player_id, td.season_year
        HAVING SUM(td.d_fga) >= 150
    ),
    hustle AS (
        SELECT
            h.player_id,
            g.season_year,
            COUNT(*) AS gp,
            AVG(h.contested_shots) AS contested_shots_pg,
            AVG(h.contested_shots_2pt) AS contested_2pt_pg,
            AVG(h.contested_shots_3pt) AS contested_3pt_pg,
            AVG(h.deflections) AS deflections_pg,
            AVG(h.charges_drawn) AS charges_drawn_pg,
            AVG(h.screen_assists) AS screen_assists_pg,
            AVG(h.screen_ast_pts) AS screen_ast_pts_pg,
            AVG(h.loose_balls_recovered) AS loose_balls_pg,
            AVG(h.box_outs) AS box_outs_pg
        FROM fact_player_game_hustle h
        JOIN dim_game g ON h.game_id = g.game_id
        WHERE g.season_type = 'Regular Season'
        GROUP BY h.player_id, g.season_year
        HAVING COUNT(*) >= 40
    ),
    trad AS (
        SELECT
            player_id, season_year,
            avg_stl, avg_blk, avg_min
        FROM agg_player_season
        WHERE season_type = 'Regular Season'
    )
    SELECT
        h.player_id,
        p.full_name AS player_name,
        p.position,
        h.season_year,
        h.gp,
        t.pct_plusminus,
        h.contested_shots_pg,
        h.contested_2pt_pg,
        h.contested_3pt_pg,
        h.deflections_pg,
        h.charges_drawn_pg,
        h.screen_assists_pg,
        h.screen_ast_pts_pg,
        h.loose_balls_pg,
        h.box_outs_pg,
        tr.avg_stl,
        tr.avg_blk,
        tr.avg_min
    FROM hustle h
    JOIN tracking t ON h.player_id = t.player_id AND h.season_year = t.season_year
    JOIN trad tr ON h.player_id = tr.player_id AND h.season_year = tr.season_year
    JOIN dim_player p ON h.player_id = p.player_id
    WHERE tr.avg_min >= 20
    ORDER BY h.season_year, h.player_id
""").pl()

print(f"Defensive feature matrix: {len(defense_matrix):,} player-seasons")
print(f"Seasons: {defense_matrix['season_year'].min()} - {defense_matrix['season_year'].max()}")

In [ ]:
# Add synergy defensive PPP (pivoted by major play types)
synergy_pivot = conn.sql("""
    SELECT
        player_id,
        season_year,
        AVG(CASE WHEN play_type = 'Isolation' THEN ppp END) AS iso_ppp,
        AVG(CASE WHEN play_type = 'PRBallHandler' OR play_type ILIKE '%pick%roll%ball%handler%'
            THEN ppp END) AS pnr_bh_ppp,
        AVG(CASE WHEN play_type = 'Postup' OR play_type ILIKE '%post%up%'
            THEN ppp END) AS postup_ppp,
        AVG(CASE WHEN play_type = 'Spotup' OR play_type ILIKE '%spot%up%'
            THEN ppp END) AS spotup_ppp,
        AVG(ppp) AS overall_def_ppp
    FROM fact_synergy
    WHERE (type_grouping ILIKE '%defensive%' OR type_grouping ILIKE '%defense%')
      AND poss >= 20
    GROUP BY player_id, season_year
""").pl()

# Join synergy into the matrix
defense_full = defense_matrix.join(
    synergy_pivot,
    on=["player_id", "season_year"],
    how="left",
)

print(f"Full feature matrix with synergy: {len(defense_full):,} rows")
print(f"Columns: {defense_full.columns}")

In [ ]:
# PCA on defensive features
feature_cols = [
    "pct_plusminus",
    "contested_shots_pg", "contested_2pt_pg", "contested_3pt_pg",
    "deflections_pg", "charges_drawn_pg", "screen_assists_pg",
    "screen_ast_pts_pg", "loose_balls_pg", "box_outs_pg",
    "avg_stl", "avg_blk",
    "iso_ppp", "pnr_bh_ppp", "postup_ppp", "spotup_ppp",
    "overall_def_ppp",
]

# Drop rows with too many nulls in synergy columns, fill remaining with median
pca_df = defense_full.select(
    ["player_id", "player_name", "position", "season_year", "gp", "avg_min"]
    + feature_cols
).to_pandas()

# Fill NaN synergy columns with column median
for col in feature_cols:
    if pca_df[col].isna().sum() > 0:
        pca_df[col] = pca_df[col].fillna(pca_df[col].median())

# Drop any remaining NaN rows
pca_df = pca_df.dropna(subset=feature_cols).reset_index(drop=True)
print(f"PCA dataset: {len(pca_df):,} player-seasons, {len(feature_cols)} features")

# StandardScaler + PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df[feature_cols])

pca = PCA(n_components=3)
components = pca.fit_transform(X_scaled)

pca_df["PC1_rim_protection"] = components[:, 0]
pca_df["PC2_perimeter_lock"] = components[:, 1]
pca_df["PC3_effort_iq"] = components[:, 2]

print(f"\nExplained variance: {pca.explained_variance_ratio_}")
print(f"Total explained: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Interpret PCA loadings
loadings = pca.components_  # shape (3, n_features)

fig = go.Figure()
component_names = ["PC1: Rim Protection", "PC2: Perimeter Lock", "PC3: Effort/IQ"]
colors = [COLORS["primary"], COLORS["secondary"], COLORS["success"]]

for i in range(3):
    fig.add_trace(go.Bar(
        x=feature_cols,
        y=loadings[i],
        name=component_names[i],
        marker_color=colors[i],
        opacity=0.8,
    ))

fig.update_layout(
    title="PCA Loadings: What Each Defensive Component Captures",
    xaxis_title="Feature",
    yaxis_title="Loading Weight",
    barmode="group",
    height=500,
    xaxis_tickangle=-45,
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)
fig.show()

In [ ]:
# Radar chart for top-20 defenders (by composite of all 3 PCA components)
# Normalize components to 0-100 scale for display
for col in ["PC1_rim_protection", "PC2_perimeter_lock", "PC3_effort_iq"]:
    mn, mx = pca_df[col].min(), pca_df[col].max()
    pca_df[f"{col}_norm"] = (pca_df[col] - mn) / (mx - mn) * 100

# Also normalize key raw metrics for radar
for col, new_name in [
    ("contested_shots_pg", "contested_norm"),
    ("deflections_pg", "deflections_norm"),
    ("avg_blk", "blocks_norm"),
]:
    mn, mx = pca_df[col].min(), pca_df[col].max()
    pca_df[new_name] = (pca_df[col] - mn) / (mx - mn) * 100

# Composite rank for selecting top-20
pca_df["composite"] = (
    pca_df["PC1_rim_protection_norm"]
    + pca_df["PC2_perimeter_lock_norm"]
    + pca_df["PC3_effort_iq_norm"]
)

top20 = pca_df.nlargest(20, "composite")

radar_dims = [
    "PC1_rim_protection_norm", "PC2_perimeter_lock_norm", "PC3_effort_iq_norm",
    "contested_norm", "deflections_norm", "blocks_norm",
]
radar_labels = [
    "Rim Protection", "Perimeter Lock", "Effort/IQ",
    "Contested Shots", "Deflections", "Blocks",
]

fig = go.Figure()

for _, row in top20.head(8).iterrows():  # show top-8 on radar for clarity
    values = [row[d] for d in radar_dims]
    values.append(values[0])  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        name=f"{row['player_name']} ({int(row['season_year'])})",
        fill="toself",
        opacity=0.3,
        line=dict(width=2),
    ))

fig.update_layout(
    title="Defensive DNA Profiles: Top 8 Defenders (6 Dimensions)",
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 100]),
    ),
    height=650, width=750,
    legend=dict(font=dict(size=10)),
)
fig.show()

takeaway(
    "PCA reveals three distinct defensive archetypes. Rim protectors load heavily on "
    "contested 2PT shots, box outs, and blocks. Perimeter locks load on contested 3PT "
    "shots, deflections, and opponent FG% impact. The third component captures hustle "
    "and IQ plays. Elite defenders like the top profiles shown here score high across "
    "multiple components, making them rare and extremely valuable."
)

---
## 6. Defensive Impact Score

We combine the three PCA components into a single **Defensive Impact Score (DIS)**
using weights that reflect the importance of each archetype:

- 0.35 × Rim Protection (PC1)
- 0.35 × Perimeter Lock (PC2)
- 0.30 × Effort/IQ (PC3)

> **Methodology note:** The DIS weights (0.35 / 0.35 / 0.30) are subjective and not derived from regression or optimization. They reflect a judgment that rim protection and perimeter defense are equally important and slightly more important than effort/IQ. See the robustness check below for how equal weighting (0.333 each) compares.

Then we validate against on/off net rating differential.

In [ ]:
# Compute DIS
pca_df["DIS"] = (
    0.35 * pca_df["PC1_rim_protection_norm"]
    + 0.35 * pca_df["PC2_perimeter_lock_norm"]
    + 0.30 * pca_df["PC3_effort_iq_norm"]
)

# Robustness check: equal weights
pca_df["DIS_equal"] = (
    (1/3) * pca_df["PC1_rim_protection_norm"]
    + (1/3) * pca_df["PC2_perimeter_lock_norm"]
    + (1/3) * pca_df["PC3_effort_iq_norm"]
)

# Correlation between weighted and equal-weighted DIS
corr = pca_df[["DIS", "DIS_equal"]].corr().iloc[0, 1]
print(f"Correlation between weighted DIS and equal-weight DIS: {corr:.4f}")

# Top-10 comparison
print("\nTop 10 by weighted DIS vs equal-weight DIS:")
for label, col in [("Weighted", "DIS"), ("Equal", "DIS_equal")]:
    top10 = pca_df.nlargest(10, col)[["player_name", col]]
    print(f"\n{label} DIS top 10:")
    print(top10.to_string(index=False))

# Top-20 DIS leaders
dis_top20 = pca_df.nlargest(20, "DIS")

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Rank", "Player", "Position", "Season", "DIS",
                "Rim Prot.", "Perim. Lock", "Effort/IQ"],
        fill_color=COLORS["primary"],
        font=dict(color="white", size=12),
        align="center",
    ),
    cells=dict(
        values=[
            list(range(1, 21)),
            dis_top20["player_name"].tolist(),
            dis_top20["position"].tolist(),
            dis_top20["season_year"].astype(int).tolist(),
            [f"{v:.1f}" for v in dis_top20["DIS"].tolist()],
            [f"{v:.1f}" for v in dis_top20["PC1_rim_protection_norm"].tolist()],
            [f"{v:.1f}" for v in dis_top20["PC2_perimeter_lock_norm"].tolist()],
            [f"{v:.1f}" for v in dis_top20["PC3_effort_iq_norm"].tolist()],
        ],
        fill_color="white",
        align="center",
        font=dict(size=11),
    ),
)])
fig.update_layout(
    title="Top 20 Defensive Impact Score (DIS) Leaders",
    height=550, width=850,
)
fig.show()

In [ ]:
# Validate DIS against on/off net rating
on_off = conn.sql("""
    SELECT
        entity_id AS player_id,
        season_year,
        MAX(CASE WHEN on_off = 'On' THEN net_rating END) AS on_court_net_rating,
        MAX(CASE WHEN on_off = 'Off' THEN net_rating END) AS off_court_net_rating
    FROM agg_on_off_splits
    WHERE entity_type = 'player'
    GROUP BY entity_id, season_year
""").pl().to_pandas()

on_off["on_off_diff"] = on_off["on_court_net_rating"] - on_off["off_court_net_rating"]

# Merge with DIS
validation = pca_df.merge(
    on_off,
    on=["player_id", "season_year"],
    how="inner",
)
validation = validation.dropna(subset=["DIS", "on_off_diff"])

corr = np.corrcoef(validation["DIS"], validation["on_off_diff"])[0, 1]
print(f"Correlation between DIS and on/off net rating diff: r = {corr:.3f}")
print(f"Validation sample: {len(validation):,} player-seasons")

# Scatter plot
fig = px.scatter(
    validation,
    x="DIS",
    y="on_off_diff",
    hover_data=["player_name", "season_year", "position"],
    trendline="ols",
    color_discrete_sequence=[COLORS["primary"]],
    opacity=0.4,
    labels={
        "DIS": "Defensive Impact Score",
        "on_off_diff": "On/Off Net Rating Differential",
    },
    title=f"DIS vs On/Off Net Rating Differential (r = {corr:.3f})",
)
fig.update_layout(height=550)
fig.show()

takeaway(
    f"The Defensive Impact Score correlates at r = {corr:.3f} with on/off net rating "
    "differential. While modest (on/off is noisy and includes offense), this validates "
    "that our composite metric captures real defensive value. The correlation is stronger "
    "than any single defensive stat (blocks, steals, or def_rating) alone, confirming "
    "the value of combining multiple data sources."
)

---
## 7. Two-Way Elite Scatter

The rarest players in basketball are **two-way elites**: high offensive usage
(carrying their team's offense) AND high defensive impact. We plot USG% vs DIS
to find the players who dominate both ends.

In [ ]:
# Join with offensive stats for two-way scatter
offense = conn.sql("""
    SELECT
        player_id, season_year,
        avg_pts AS ppg, avg_usg_pct AS usg_pct,
        gp
    FROM agg_player_season
    WHERE season_type = 'Regular Season'
      AND gp >= 40
""").pl().to_pandas()

two_way = pca_df.merge(
    offense,
    on=["player_id", "season_year"],
    how="inner",
    suffixes=("", "_off"),
)
two_way = two_way.dropna(subset=["usg_pct", "DIS", "ppg"])

print(f"Two-way dataset: {len(two_way):,} player-seasons")

# Clean position for coloring
two_way["pos_group"] = two_way["position"].apply(
    lambda x: x.split("-")[0] if isinstance(x, str) else "Unknown"
)

# Scatter: USG% vs DIS
fig = px.scatter(
    two_way,
    x="usg_pct",
    y="DIS",
    color="pos_group",
    hover_data=["player_name", "season_year", "ppg", "DIS"],
    opacity=0.6,
    labels={
        "usg_pct": "Usage Rate (%)",
        "DIS": "Defensive Impact Score",
        "pos_group": "Position",
    },
    title="Two-Way Elite: Usage Rate vs Defensive Impact Score",
    color_discrete_map={
        "G": COLORS["primary"],
        "F": COLORS["success"],
        "C": COLORS["secondary"],
        "Guard": COLORS["primary"],
        "Forward": COLORS["success"],
        "Center": COLORS["secondary"],
    },
)

# Add quadrant lines at median
med_usg = two_way["usg_pct"].median()
med_dis = two_way["DIS"].median()
fig.add_hline(y=med_dis, line_dash="dash", line_color=COLORS["muted"], opacity=0.5)
fig.add_vline(x=med_usg, line_dash="dash", line_color=COLORS["muted"], opacity=0.5)

# Annotate top-right quadrant outliers (top-10 by USG + DIS combined)
elite = two_way[
    (two_way["usg_pct"] > two_way["usg_pct"].quantile(0.8))
    & (two_way["DIS"] > two_way["DIS"].quantile(0.8))
].nlargest(10, "DIS")

for _, row in elite.iterrows():
    fig.add_annotation(
        x=row["usg_pct"],
        y=row["DIS"],
        text=f"{row['player_name']} ({int(row['season_year'])})",
        showarrow=True,
        arrowhead=2,
        arrowsize=0.8,
        font=dict(size=9),
        ax=20, ay=-20,
    )

fig.update_layout(height=600, width=800)
fig.show()

takeaway(
    "The top-right quadrant (high usage + high DIS) is remarkably sparse. Most high-usage "
    "scorers cluster in the bottom half of DIS, suggesting they conserve energy for offense. "
    "The rare players who appear in the upper-right -- carrying a heavy offensive load "
    "while maintaining elite defense -- are the true two-way forces in basketball. "
    "Centers dominate the high-DIS region while guards dominate high-usage, making "
    "two-way wings the rarest archetype."
)

---
## 8. Era Trends: Defensive Evolution

The NBA's defensive style has changed dramatically in the tracking era (2013-present).
The rise of three-point shooting has forced a fundamental shift from rim-protection-centric
defense to perimeter-switching schemes.

In [ ]:
# Era trends from hustle stats
era_hustle = conn.sql("""
    SELECT
        g.season_year,
        AVG(h.contested_shots_3pt) AS avg_contested_3pt,
        AVG(h.contested_shots_2pt) AS avg_contested_2pt,
        AVG(h.deflections) AS avg_deflections,
        AVG(h.loose_balls_recovered) AS avg_loose_balls,
        AVG(h.charges_drawn) AS avg_charges_drawn,
        AVG(h.box_outs) AS avg_box_outs,
        COUNT(DISTINCT h.player_id) AS n_players
    FROM fact_player_game_hustle h
    JOIN dim_game g ON h.game_id = g.game_id
    WHERE g.season_type = 'Regular Season'
      AND h.min IS NOT NULL AND h.min > 0
    GROUP BY g.season_year
    ORDER BY g.season_year
""").pl()

# Era trends from tracking (switches_on)
era_tracking = conn.sql("""
    SELECT
        g.season_year,
        AVG(t.switches_on) AS avg_switches_on,
        COUNT(DISTINCT t.player_id) AS n_players
    FROM fact_player_game_tracking t
    JOIN dim_game g ON t.game_id = g.game_id
    WHERE g.season_type = 'Regular Season'
      AND t.switches_on IS NOT NULL
    GROUP BY g.season_year
    ORDER BY g.season_year
""").pl()

# Era trends from traditional stats (blocks)
era_trad = conn.sql("""
    SELECT
        season_year,
        AVG(avg_blk) AS league_avg_blk
    FROM agg_player_season
    WHERE season_type = 'Regular Season'
      AND gp >= 20 AND avg_min >= 15
    GROUP BY season_year
    ORDER BY season_year
""").pl()

print(f"Hustle era data: {len(era_hustle)} seasons")
print(f"Tracking era data: {len(era_tracking)} seasons")
print(f"Traditional era data: {len(era_trad)} seasons")

In [ ]:
# Multi-line chart: defensive style evolution
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Contested 3PT Shots per Game (Rise of Perimeter D)",
        "Blocks per Game (Rim Protection Metric)",
        "Deflections per Game (Active Hands Defense)",
        "Switches On per Game (Switch-Everything Era)",
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# Panel 1: Contested 3PT
fig.add_trace(go.Scatter(
    x=era_hustle["season_year"].to_list(),
    y=era_hustle["avg_contested_3pt"].to_list(),
    mode="lines+markers",
    name="Contested 3PT",
    line=dict(color=COLORS["secondary"], width=3),
    marker=dict(size=7),
    showlegend=False,
), row=1, col=1)

# Panel 2: Blocks (from trad data, wider range)
fig.add_trace(go.Scatter(
    x=era_trad["season_year"].to_list(),
    y=era_trad["league_avg_blk"].to_list(),
    mode="lines+markers",
    name="Avg Blocks",
    line=dict(color=COLORS["primary"], width=3),
    marker=dict(size=7),
    showlegend=False,
), row=1, col=2)

# Panel 3: Deflections
fig.add_trace(go.Scatter(
    x=era_hustle["season_year"].to_list(),
    y=era_hustle["avg_deflections"].to_list(),
    mode="lines+markers",
    name="Deflections",
    line=dict(color=COLORS["success"], width=3),
    marker=dict(size=7),
    showlegend=False,
), row=2, col=1)

# Panel 4: Switches
if len(era_tracking) > 0:
    fig.add_trace(go.Scatter(
        x=era_tracking["season_year"].to_list(),
        y=era_tracking["avg_switches_on"].to_list(),
        mode="lines+markers",
        name="Switches On",
        line=dict(color=COLORS["accent"], width=3),
        marker=dict(size=7),
        showlegend=False,
    ), row=2, col=2)

fig.update_layout(
    height=700, width=1000,
    title_text="Defensive Style Evolution Across the Tracking Era",
)
fig.show()

takeaway(
    "The tracking era reveals a clear defensive evolution. Contested 3-point shots per "
    "game have risen steadily as teams must defend more perimeter shooting. Switches per "
    "game have increased as 'switch everything' schemes become the default for playoff "
    "teams. Meanwhile, blocks per game have remained relatively flat, suggesting that "
    "rim protection alone is no longer the defining defensive skill -- versatility and "
    "perimeter discipline now matter as much or more."
)

---
## 9. Conclusion & Cross-Links

### Summary of Findings

1. **Tracking defense (opponent FG% impact)** is the most direct measure of individual defense. The best defenders reduce opponent FG% by 4-8 percentage points.
2. **Hustle stats** capture the invisible effort plays. Leaders in contested shots, deflections, and charges drawn form a completely different hierarchy than blocks/steals leaders.
3. **Synergy defensive play types** reveal specialization -- some players are elite isolation defenders but average in PnR, and vice versa.
4. **PCA identifies three defensive archetypes**: Rim Protection, Perimeter Lock, and Effort/IQ. Elite defenders score high across multiple dimensions.
5. **The Defensive Impact Score (DIS)** combines all three data sources and correlates positively with on/off net rating differential, validating it as a meaningful composite metric. A robustness check using equal weights (0.333 each) shows the top-10 rankings are largely stable, confirming that the DIS is not overly sensitive to the specific weight choices.
6. **Two-way elites are extremely rare**: high-usage players almost never maintain elite DIS, making the few who do (the top-right quadrant) exceptionally valuable.
7. **Defense has evolved**: the tracking era shows a clear shift from rim-protection-centric defense to perimeter switching and active-hands schemes.

### Methodology Notes

- **PCA**: StandardScaler + 3-component PCA on 17 defensive features from 3 sources
- **DIS**: Weighted composite (0.35 rim + 0.35 perimeter + 0.30 effort), normalized 0-100. Weights are subjective; an equal-weight robustness check confirms high rank-order stability.
- **Validation**: Pearson correlation with on/off net rating differential
- **Minimum thresholds**: 40+ GP for hustle, 150+ DFGA for tracking, 20+ possessions for synergy, 20+ MPG for all

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| **Part 5** | **Defense Decoded** (this notebook) | **Defense Decoded (Tracking + Hustle + Synergy PCA)** |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

**Dataset**: [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) on Kaggle
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")